# Special topic: graph or not?

A graph index costs thousands of model calls to build. This notebook asks three questions before you
pay that bill, each on data this repository already holds:

1. Is the graph's gain on multi-hop questions a gain over a weak retriever? RI #4's `hipporag2` beats
   a `bge-small-en-v1.5` dense retriever by 0.140 F1 on 300 MuSiQue questions. Here the same questions
   are ranked by a stack of local models (dense, SPLADE, ColBERT,
   and the `bge-reranker-v2-m3` cross-encoder) and answered by the same reader.
2. What does a governed graph answer that retrieval over documents cannot? The RI #4 ServerFault graph
   (7 entity types, 7 relation types, a citation per fact) is asked counts, complete lists, relations
   across posts, and paths between entities that no single post links.
3. Does your corpus look like one where a graph pays? A check with no model calls reports how often
   names recur across documents, how many questions are relational, how many need a document that
   names nothing in the question, and how much headroom dense retrieval leaves.

Every run here is one run per arm with no repeats and no seeds, so treat a difference under about
10 points as noise. The notebook reads the committed results unless you set `RUN_EXPENSIVE = True`.

In [1]:
import json
import sys
from pathlib import Path

from aieng.advanced_rag.models import choose_models


sys.path.insert(0, str(Path.cwd()))
from gon import readiness  # noqa: E402


RUN_EXPENSIVE = False
MODELS = choose_models(show=True)
OUTPUTS = Path("outputs")

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget not set on the key, spent $85.72


## Strong local retrieval against HippoRAG 2 on MuSiQue

The reader is RI #4's: `Qwen3_6-35B-A3B` answers each question from an arm's top five passages with
RI #4's prompt, and `gemini-3.8-flash` judges each answer given the question, the reference, and the
five passages. The `hipporag2` and `dense` rows are RI #4's committed rows, compared question by
question with a paired bootstrap on F1.

The arms:

- `hybrid3`: reciprocal rank fusion of `bge-small-en-v1.5`, `Splade_PP_en_v1`, and
  `answerai-colbert-small-v1`.
- `hybrid3-rerank`: `bge-reranker-v2-m3` over the top 100 of `hybrid3`.
- `chain-rerank`: `hybrid3-rerank`, then a second retrieval for each of its top two passages with the
  question plus that passage's text, interleaved. A bridge step with no graph and no model call.
- `union-rerank`: the cross-encoder over `hipporag2`'s top 10 together with `hybrid3`'s top 100.
- `fusion`: reciprocal rank fusion of `hybrid3-rerank` and `hipporag2`.

`RUN_EXPENSIVE = True` reruns the arms: ranking needs `torch` (a GPU takes minutes, a CPU hours), and
answering makes 1,500 on-prem calls.

In [2]:
if RUN_EXPENSIVE:
    import run_graph_or_not

    run_graph_or_not.main(["x1"])

x1 = json.loads((OUTPUTS / "x1_musique.json").read_text())
ri4 = json.loads(Path("../../graphrag_technical_support/outputs/musique_graph_test.json").read_text())


def row(name, arm):
    """Return one markdown table row for an arm."""
    s = arm["summary"]["all"]
    judged = arm.get("judged", {}).get("all")
    return (
        f"| `{name}` | {s['all_supporting_recall_at_5']:.3f} | {s['all_supporting_recall_at_10']:.3f} "
        f"| {s['exact_match']:.3f} | {s['f1']:.3f} | {'' if judged is None else f'{judged:.3f}'} |"
    )


lines = ["| Arm | All-supporting @5 | @10 | EM | F1 | Judged |", "|---|---|---|---|---|---|"]
lines += [row(n, ri4["arms"][n]) for n in ("dense", "hipporag2")]
lines += [row(n, a) for n, a in x1["arms"].items()]
print("\n".join(lines))

| Arm | All-supporting @5 | @10 | EM | F1 | Judged |
|---|---|---|---|---|---|
| `dense` | 0.163 | 0.237 | 0.193 | 0.250 | 0.243 |
| `hipporag2` | 0.320 | 0.390 | 0.297 | 0.390 | 0.377 |
| `hybrid3` | 0.217 | 0.303 | 0.240 | 0.310 | 0.317 |
| `hybrid3-rerank` | 0.233 | 0.283 | 0.260 | 0.338 | 0.333 |
| `chain-rerank` | 0.290 | 0.387 | 0.317 | 0.388 | 0.393 |
| `union-rerank` | 0.233 | 0.290 | 0.260 | 0.338 | 0.333 |
| `fusion` | 0.327 | 0.433 | 0.303 | 0.381 | 0.373 |


In [3]:
for name, comparison in x1["comparisons"].items():
    if name.endswith("_vs_hipporag2"):
        c = comparison["all"]
        low, high = c["f1_interval"]
        recall = c["all_supporting_at_10"]
        print(
            f"{name:28s} F1 {c['f1_difference']:+.3f} (95% interval {low:+.3f} to {high:+.3f}); "
            f"all supporting @10 only here {recall['only_candidate']}, only hipporag2 {recall['only_baseline']}"
        )

hybrid3_vs_hipporag2         F1 -0.080 (95% interval -0.129 to -0.034); all supporting @10 only here 17, only hipporag2 43
hybrid3-rerank_vs_hipporag2  F1 -0.052 (95% interval -0.103 to -0.001); all supporting @10 only here 19, only hipporag2 51
chain-rerank_vs_hipporag2    F1 -0.003 (95% interval -0.053 to +0.047); all supporting @10 only here 29, only hipporag2 30
union-rerank_vs_hipporag2    F1 -0.052 (95% interval -0.103 to -0.001); all supporting @10 only here 19, only hipporag2 49
fusion_vs_hipporag2          F1 -0.009 (95% interval -0.045 to +0.027); all supporting @10 only here 18, only hipporag2 5


In [4]:
print("Retrieval only (no answers):")
for name, table in x1["retrieval_only"].items():
    print(
        f"  {name:24s} all-supporting @5 {table['all_supporting_recall_at_5']:.3f}  @10 {table['all_supporting_recall_at_10']:.3f}"
    )

Retrieval only (no answers):
  splade                   all-supporting @5 0.163  @10 0.243
  colbert                  all-supporting @5 0.217  @10 0.300
  hybrid3-rerank-minilm    all-supporting @5 0.217  @10 0.287
  dense                    all-supporting @5 0.163  @10 0.237


## Relational questions over the ServerFault graph

Sixty questions, fifteen per family, generated from graph patterns with a fixed seed and no model
call. `count` and `co_mention` questions ask which posts mention a name, so they have two golds: what
the graph returns, and the posts whose text contains the name (`lexical`), which does not depend on
the graph. `relation` and `path` questions have only the graph gold, so a hand check of their facts
against the posts bounds what that gold is worth.

The arms read the same 9,999 posts or the same graph:

- `dense` reads the top 10 posts by `bge-small-en-v1.5`; `dense-map` asks the reader about each of the
  top 50 posts in turn and combines the replies.
- `grep` finds the posts that contain the names; `grep-map` asks the reader about each of them.
- `graph-template` runs a fixed SQL query per family over the graph's tables; `graph-like` matches
  entity names by substring, so `sysctl` also finds `sysctl.conf`.
- `text2sql` has the reader write the query from the schema; `text2sql-repair` gives it one retry when
  the query fails or returns nothing.

In [5]:
if RUN_EXPENSIVE:
    run_graph_or_not.main(["x3"])

x3 = json.loads((OUTPUTS / "x3_serverfault.json").read_text())
families = ("count", "co_mention", "relation", "path")
header = "| Arm | count (graph / lexical) | co_mention F1 (graph / lexical) | relation F1 | path F1 |"
lines = [header, "|---|---|---|---|---|"]
for arm, cells in x3["summary"].items():
    c, m, r, p = (cells[f] for f in families)
    lines.append(
        f"| `{arm}` | {c['graph_within_20pct']:.3f} / {c['lexical_within_20pct']:.3f} "
        f"| {m['graph_f1']:.3f} / {m['lexical_f1']:.3f} | {r['graph_f1']:.3f} | {p['graph_f1']:.3f} |"
    )
print("\n".join(lines))
print("count cells: share of questions answered within 20% of the gold count")

| Arm | count (graph / lexical) | co_mention F1 (graph / lexical) | relation F1 | path F1 |
|---|---|---|---|---|
| `dense` | 0.000 / 0.000 | 0.313 / 0.237 | 0.089 | 0.073 |
| `dense-map` | 0.133 / 0.067 | 0.320 / 0.380 | 0.135 | 0.015 |
| `grep` | 0.200 / 1.000 | 0.796 / 1.000 | 0.226 | 0.089 |
| `grep-map` | 0.200 / 0.800 | 0.736 / 0.919 | 0.179 | 0.000 |
| `graph-template` | 1.000 / 0.200 | 1.000 / 0.796 | 1.000 | 1.000 |
| `graph-like` | 0.333 / 0.600 | 0.907 / 0.859 | 1.000 | 1.000 |
| `text2sql` | 1.000 / 0.200 | 1.000 / 0.796 | 0.727 | 0.067 |
| `text2sql-repair` | 1.000 / 0.200 | 1.000 / 0.796 | 0.851 | 0.067 |
count cells: share of questions answered within 20% of the gold count


In [6]:
hand = x3["hand_check"]
print(
    f"relation facts supported by their post: {hand['relation_facts']['supported']} of {hand['relation_facts']['checked']}"
)
print(
    f"posts grep found and the graph's exact name missed that do mention the name: "
    f"{hand['lexical_only_mentions']['real_mentions']} of {hand['lexical_only_mentions']['checked']}"
)
coverage = x3["graph_coverage"]
print(
    f"graph exact-name recall of lexical mentions on count questions: {coverage['exact_name_recall']:.2f}; "
    f"with substring match: {coverage['substring_recall']:.2f}"
)

relation facts supported by their post: 12 of 20
posts grep found and the graph's exact name missed that do mention the name: 14 of 14
graph exact-name recall of lexical mentions on count questions: 0.37; with substring match: 0.63


## Check your own corpus before building a graph

`readiness.graph_readiness` makes no model call. Give it your documents, and if you have them, your
questions, the documents that answer them, and a search function. Without an extractor's names it
uses capitalised spans and code-like tokens (`sysctl.conf`, `/etc/hosts`) as candidate names. The
table below is the reference run on the corpora in this repository.

In [7]:
x2 = json.loads((OUTPUTS / "x2_readiness.json").read_text())
keys = (
    "recurring_name_share",
    "linked_document_share",
    "relational_question_share",
    "bridge_question_share",
    "dense_all_supporting_at_k",
)
print("| Corpus / questions / names | " + " | ".join(keys) + " |")
print("|---|" + "---|" * len(keys))
for name, r in x2["rows"].items():
    cells = ["" if r.get(k) is None else f"{r[k]:.2f}" for k in keys]
    print(f"| {name} | " + " | ".join(cells) + " |")

| Corpus / questions / names | recurring_name_share | linked_document_share | relational_question_share | bridge_question_share | dense_all_supporting_at_k |
|---|---|---|---|---|---|
| musique / multi-hop questions / graph names | 0.26 | 1.00 | 0.11 | 0.45 | 0.24 |
| musique / multi-hop questions / regex names | 0.27 | 1.00 | 0.11 | 0.48 | 0.24 |
| serverfault / support questions / graph names | 0.40 | 1.00 | 0.00 | 0.17 | 0.93 |
| serverfault / support questions / regex names | 0.29 | 1.00 | 0.00 | 0.07 | 0.93 |
| serverfault / X3 relational questions / graph names | 0.40 | 1.00 | 1.00 |  |  |
| contractnli / hypotheses / regex names | 0.27 | 1.00 | 0.00 |  |  |


In [8]:
# Your documents and questions go here. This makes no model call.
my_documents = {
    "runbook-1": "Restart php-fpm after editing /etc/php/8.3/fpm/pool.d/www.conf. Nginx returns 502 until it is back.",
    "runbook-2": "Nginx proxies to php-fpm over a Unix socket. SELinux can block the socket.",
    "runbook-3": "Postfix relays outbound mail through the smarthost.",
}
my_questions = {
    "q1": "Which services does Nginx depend on?",
    "q2": "How many runbooks mention php-fpm?",
}
report = readiness.graph_readiness(my_documents, questions=my_questions, hub_share=1.0)
for line in report.reading():
    print("-", line)

- 17% of names recur across documents; 67% of documents share a name with another.
- 100% of questions ask for counts, lists, comparisons, or paths (top-k retrieval cannot guarantee these are complete).
